In [1]:
import math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl

torch.set_float32_matmul_precision("medium")  # lets Torch use Tensor Cores on your 3050


# metrics (meters)
def ade(pred, gt, mask=None):
    # pred, gt: [B,T,2]
    err = torch.linalg.vector_norm(pred - gt, dim=-1)  # [B,T]
    if mask is not None:
        err = err * mask
        return (err.sum(dim=1) / mask.sum(dim=1).clamp_min(1)).mean()
    return err.mean()

def fde(pred, gt, mask=None):
    err = torch.linalg.vector_norm(pred[:, -1] - gt[:, -1], dim=-1)  # [B]
    if mask is not None:
        return (err * mask[:, -1]).sum() / mask[:, -1].sum().clamp_min(1)
    return err.mean()


c:\Users\fiona\Documents\GitHub\Transformers\transformers-mini\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class GRUTraj(pl.LightningModule):
    """
    Agent-only trajectory forecaster:
      inputs:  past positions (rel to last observed) [B,20,2]
      outputs: future positions (rel)                [B,30,2]
    """
    def __init__(
        self,
        obs_len=20,
        pred_len=30,
        hidden_size=128,
        num_layers=1,
        dropout=0.0,
        lr=1e-3,
        loss_type="l1",          # "l1" or "mse" or "huber"
        teacher_forcing=0.7,     # probability per step during training
        tf_decay_epochs=10,      # linear decay to 0 over this many epochs
        weight_decay=0.0,
        grad_clip=1.0,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.obs_len = obs_len
        self.pred_len = pred_len
        self.tf0 = teacher_forcing
        self.tf_decay_epochs = tf_decay_epochs

        self.encoder = nn.GRU(
            input_size=2, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0, batch_first=True
        )
        self.decoder = nn.GRU(
            input_size=2, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0, batch_first=True
        )
        self.head = nn.Linear(hidden_size, 2)

        if loss_type == "l1":
            self.crit = nn.L1Loss()
        elif loss_type == "mse":
            self.crit = nn.MSELoss()
        elif loss_type == "huber":
            self.crit = nn.HuberLoss(delta=1.0)
        else:
            raise ValueError("loss_type must be 'l1', 'mse', or 'huber'")

        self.grad_clip = grad_clip

    # --- utils ---
    def encode(self, past):              # past: [B,obs,2]
        _, h = self.encoder(past)        # h: [L,B,H]
        return h

    def _decode_autoregressive(self, h, start, steps):
        prev = start.unsqueeze(1)        # [B,1,2]
        outs = []
        for _ in range(steps):
            y, h = self.decoder(prev, h) # y: [B,1,H]
            y = self.head(y)             # [B,1,2]
            outs.append(y)
            prev = y
        return torch.cat(outs, dim=1)    # [B,T,2]

    def _decode_teacher_forced(self, h, start, future, tf_ratio):
        B, T, _ = future.shape
        prev = start.unsqueeze(1)
        outs = []
        tf_mask = torch.rand(T, device=future.device) < tf_ratio
        for t in range(T):
            y, h = self.decoder(prev, h)
            y = self.head(y)             # [B,1,2]
            outs.append(y)
            prev = future[:, t:t+1, :] if tf_mask[t] else y.detach()
        return torch.cat(outs, dim=1)

    def current_tf_ratio(self):
        if self.tf_decay_epochs <= 0:
            return self.tf0
        e = getattr(self, "current_epoch", 0)
        return float(max(0.0, self.tf0 * (1.0 - e / self.tf_decay_epochs)))

    # --- lightning hooks ---
    def training_step(self, batch, _):
        past   = batch["past"].float()          # [B,20,2]
        future = batch["future"].float()        # [B,30,2]
        h = self.encode(past)
        start = torch.zeros(past.size(0), 2, device=past.device, dtype=past.dtype)
        tf_ratio = self.current_tf_ratio()
        pred = self._decode_teacher_forced(h, start, future, tf_ratio)

        loss = self.crit(pred, future)
        self.log("train/loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log("train/ade", ade(pred, future), prog_bar=False, on_epoch=True)
        self.log("train/fde", fde(pred, future), prog_bar=False, on_epoch=True)
        self.log("train/tf_ratio", tf_ratio, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, _):
        past   = batch["past"].float()
        future = batch["future"].float()        # val should have labels
        h = self.encode(past)
        start = torch.zeros(past.size(0), 2, device=past.device, dtype=past.dtype)
        pred = self._decode_autoregressive(h, start, self.pred_len)
        loss = self.crit(pred, future)
        self.log("val/loss", loss, prog_bar=True)
        self.log("val/ade", ade(pred, future), prog_bar=True)
        self.log("val/fde", fde(pred, future), prog_bar=True)

    def test_step(self, batch, _):
        # test may not have labels
        past = batch["past"].float()
        h = self.encode(past)
        start = torch.zeros(past.size(0), 2, device=past.device, dtype=past.dtype)
        pred = self._decode_autoregressive(h, start, self.pred_len)
        return {"pred": pred, "path": batch["path"]}

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "monitor": "val/loss"}}


In [2]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset

PRED_LEN = 30  # only used to make a placeholder if you ever want one

class AgentNPZDataset(Dataset):
    """
    Loads per-scene NPZ files written by your preprocessor.
    - Train/Val: include_future=True  -> returns 'future'
    - Test:      include_future=False -> omits 'future' (clean inference)
    Always returns: 'past', 'origin', 'path', and 'has_future' (bool)
    """
    def __init__(self, npz_dir: Path, include_future: bool = True):
        self.dir = Path(npz_dir)
        self.files = sorted(self.dir.glob("*.npz"))
        assert self.files, f"No .npz files in {self.dir}"
        self.include_future = include_future

    def __len__(self): 
        return len(self.files)

    def __getitem__(self, i):
        f = self.files[i]
        with np.load(f, mmap_mode="r") as z:
            past   = torch.from_numpy(z["past"])     # (20,2)
            origin = torch.from_numpy(z["origin"])   # (2,)
            path   = str(z["path"])
            has_future = bool(int(z["has_future"])) if "has_future" in z.files else True

            sample = {"past": past, "origin": origin, "path": path, "has_future": torch.tensor(has_future)}

            if self.include_future:
                # For train/val we include future; if a test file sneaks in with no future,
                # we still return a tensor (zeros) so batch collation works.
                if has_future and "future" in z.files:
                    fut = torch.from_numpy(z["future"])  # (30,2)
                else:
                    fut = torch.zeros((PRED_LEN, 2), dtype=past.dtype)
                sample["future"] = fut

            return sample


In [3]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader

class AgentNPZDataModule(pl.LightningDataModule):
    def __init__(self, root_dir="preprocessed/agent_only_npz", batch_size=256, num_workers=4,
                 include_future_in_test: bool = False):
        super().__init__()
        self.root_dir = Path(root_dir)
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.include_future_in_test = include_future_in_test
        self.splits = {}
        self._train_dl = self._val_dl = self._test_dl = None

    def setup(self, stage=None):
        for split in ("train","val","test"):
            split_dir = self.root_dir / split
            if split_dir.exists() and list(split_dir.glob("*.npz")):
                include_future = not (split == "test" and not self.include_future_in_test)
                self.splits[split] = AgentNPZDataset(split_dir, include_future=include_future)
            else:
                print(f"[DataModule] Split '{split}' not available at {split_dir} — skipping.")

        # build and cache the loaders ONCE so workers stay alive across calls
        def _dl(ds, shuffle):
            return DataLoader(
                ds, batch_size=self.batch_size, shuffle=shuffle,
                num_workers=self.num_workers, pin_memory=True,
                persistent_workers=self.num_workers>0, prefetch_factor=2
            )

        if "train" in self.splits and self._train_dl is None:
            self._train_dl = _dl(self.splits["train"], True)
        if "val" in self.splits and self._val_dl is None:
            self._val_dl = _dl(self.splits["val"], False)
        if "test" in self.splits and self._test_dl is None:
            self._test_dl = _dl(self.splits["test"], False)

    def train_dataloader(self): return self._train_dl
    def val_dataloader(self):   return self._val_dl
    def test_dataloader(self):  return self._test_dl


c:\Users\fiona\Documents\GitHub\Transformers\transformers-mini\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import time, os
import pytorch_lightning as pl
from pytorch_lightning.callbacks import TQDMProgressBar, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

# If you've already created dm elsewhere, you can reuse it; otherwise:
dm = AgentNPZDataModule(
    root_dir="preprocessed/agent_only_npz",
    batch_size=512,
    num_workers=max(1, (os.cpu_count() or 2)//2),
    include_future_in_test=False  # test batches won't include 'future'
)
dm.setup()

model = GRUTraj(
    obs_len=20, pred_len=30,
    hidden_size=128, num_layers=1, dropout=0.0,
    lr=1e-3, loss_type="l1", teacher_forcing=0.7, tf_decay_epochs=10,
    weight_decay=0.0, grad_clip=1.0
)

# 1) progress bar that refreshes every step
pbar = TQDMProgressBar(refresh_rate=1)

# 2) CSV logs you can tail in the notebook
csv_logger = CSVLogger(save_dir="lightning_logs", name="gru_agent")

# 3) print samples/sec + GPU memory every N steps
class SpeedCallback(pl.Callback):
    def __init__(self, every_n=50): self.every_n = every_n; self.t0 = None
    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        if batch_idx % self.every_n == 0: self.t0 = time.time()
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if (batch_idx + 1) % self.every_n == 0:
            bsz = int(batch["past"].size(0))
            dt = max(1e-6, time.time() - (self.t0 or time.time()))
            speed = (self.every_n * bsz) / dt
            mem = (torch.cuda.memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
            # 'train/loss' is logged by your module; show if available
            loss = trainer.logged_metrics.get("train/loss")
            loss_str = f" loss={float(loss):.3f}" if loss is not None else ""
            trainer.print(f"[ep {trainer.current_epoch:02d} step {batch_idx+1:05d}] {speed:7.1f} samples/s | gpu_mem={mem:.2f} GB{loss_str}")

speed_cb = SpeedCallback(every_n=50)
lrmon = LearningRateMonitor(logging_interval="epoch")
ckpt  = ModelCheckpoint(monitor="val/loss", mode="min", save_top_k=1, save_last=True,
                        filename="gru-ao-{epoch:02d}-{val_loss:.4f}")

trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    num_sanity_val_steps=0,      # skip slow pre-val
    gradient_clip_val=model.grad_clip,
    log_every_n_steps=10,        # how often to push metrics to logger/progress bar
    callbacks=[pbar, lrmon, ckpt, speed_cb],
    logger=csv_logger,
)

"""trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    precision="16-mixed",          # fast + stable on RTX 30-series
    num_sanity_val_steps=0,        # <-- skip slow pre-val
    gradient_clip_val=model.grad_clip,
    log_every_n_steps=20,
)"""



Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


'trainer = pl.Trainer(\n    max_epochs=5,\n    accelerator="auto",\n    devices=1,\n    precision="16-mixed",          # fast + stable on RTX 30-series\n    num_sanity_val_steps=0,        # <-- skip slow pre-val\n    gradient_clip_val=model.grad_clip,\n    log_every_n_steps=20,\n)'

In [ ]:
trainer.fit(model, datamodule=dm)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\fiona\Documents\GitHub\Transformers\transformers-mini\.venv312\Lib\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name    | Type   | Params | Mode 
-------------------------------------------
0 | encoder | GRU    | 50.7 K | train
1 | decoder | GRU    | 50.7 K | train
2 | head    | Linear | 258    | train
3 | crit    | L1Loss | 0      | train
-------------------------------------------
101 K     Trainable params
0         Non-trainable params
101 K     Total params
0.407     Total estimated model params size (MB)
4         Modules in train mode
0         Modules in eval mode


In [ ]:
import matplotlib.pyplot as plt

model.eval()
dl = dm.val_dataloader() or dm.train_dataloader()
batch = next(iter(dl))

with torch.no_grad():
    pred = model(batch["past"].float().to(model.device)).cpu()  # [B,30,2]

i = 0  # pick an example
past  = batch["past"][i].numpy()
fut   = batch.get("future")[i].numpy() if "future" in batch else None
predi = pred[i].numpy()

plt.figure(figsize=(4,4))
plt.plot(past[:,0], past[:,1], lw=2, label="past")
if fut is not None:
    plt.plot(fut[:,0], fut[:,1], lw=2, label="future GT")
plt.plot(predi[:,0], predi[:,1], lw=2, label="pred")
plt.scatter([0],[0], c="k", s=30, marker="x", label="origin")
plt.gca().set_aspect("equal"); plt.grid(True, alpha=0.3); plt.legend()
plt.title("Agent-only GRU baseline (relative coords)")
plt.show()
